## Experiment to identify memory capacity in the RNN hidden state

### Recipe: experiment 1
- generate completely random sequence
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

### Recipe: experiment 2
- generate a sequence with pattern (1,2,3,4,5, 1,2,3,4,5, 1,2,3,4,5....)
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

In [1]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 

In [2]:
def get_random_sequence(total_samples, token_number=7):
    return np.vectorize(chr)(np.random.randint(token_number, size=total_samples) + ord('A'))

In [3]:
# tokens start from 1
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-64
      
            self.y[ii] = \
                ord(data[ii+jj+1])-64

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [4]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)
        out, h = self.rnn(embedded, h)
        out = self.linear(out[:,-1,:])

        return out, h  

In [63]:
class classifier(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.linear1 = nn.Linear(hidden_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        out = nn.functional.relu(self.linear1(x))
        out = self.linear2(x)

        return out  

In [65]:
### initial training ###
total_samples = 30000
working_memory = 1
short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 1e-3
test_acc = []

model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95, weight_decay=1e-8)
criterion = torch.nn.CrossEntropyLoss()

data = get_random_sequence(total_samples, token_number=vocab_size-1)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        y_pred, h = model(X)
    else:
        y_pred, h = model(X, hidden)

    loss = criterion(y_pred, y[0])     
    loss.backward()
    optimizer.step()

    # print(total)
    with torch.no_grad():
        hidden = h.clone()
        total += 1

        if y[0] == y_pred[0].argmax():
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0


        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')
    
        

Iter : 1001, loss: 2.0470, accuracy: 0.1490
Iter : 2001, loss: 2.0877, accuracy: 0.1300
Iter : 3001, loss: 2.0220, accuracy: 0.1480
Iter : 4001, loss: 1.7790, accuracy: 0.1530
Iter : 5001, loss: 1.9778, accuracy: 0.1370
Iter : 6001, loss: 2.2457, accuracy: 0.1240
Iter : 7001, loss: 1.9831, accuracy: 0.1430
Iter : 8001, loss: 1.9084, accuracy: 0.1450
Iter : 9001, loss: 2.2098, accuracy: 0.1460
Iter : 10001, loss: 2.1444, accuracy: 0.1460
Iter : 11001, loss: 1.9490, accuracy: 0.1570
Iter : 12001, loss: 1.6896, accuracy: 0.1540
Iter : 13001, loss: 1.9646, accuracy: 0.1390
Iter : 14001, loss: 2.1487, accuracy: 0.1510
Iter : 15001, loss: 1.9958, accuracy: 0.1510
Iter : 16001, loss: 1.8227, accuracy: 0.1410
Iter : 17001, loss: 1.9579, accuracy: 0.1480
Iter : 18001, loss: 1.8778, accuracy: 0.1190
Iter : 19001, loss: 2.1572, accuracy: 0.1600
Iter : 20001, loss: 1.9597, accuracy: 0.1630
Iter : 21001, loss: 1.8121, accuracy: 0.1400
Iter : 22001, loss: 2.1425, accuracy: 0.1650
Iter : 23001, loss:

In [66]:
# tokens start from 1
class Dataset_reconstructer(Dataset):
    def __init__(self, hidden_states, data, past_recall=1):
        
        self.X = np.array(hidden_states)
        self.y = np.vectorize(ord)(data) - 64
        self. y = np.concatenate(
                (np.zeros(past_recall, dtype=int), self.y[:-past_recall])
            )

        self.X = tnsr(self.X)
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [67]:
hidden_states = []

for X, y in train_loader:
    with torch.no_grad():
        if total == 0:
            y_pred, h = model(X)
        else:
            y_pred, h = model(X, h)

        hidden_states.append(
            h[0][0]
        )

In [68]:
data_set = Dataset_reconstructer(hidden_states, data, past_recall=2)
reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

In [69]:
reconstructor = classifier(vocab_size, hidden_size)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)

for epoch in range(5):
    for X, y in reconstruct_loader:
        optimizer.zero_grad()

        y_pred = reconstructor(X)
        

        loss = criterion(y_pred, y)     
        loss.backward()
        optimizer.step()

        # print(total)
        with torch.no_grad():
            total += 1

            if y == y_pred.argmax():
                    correct[total%1000] = 1
            else:
                correct[total%1000] = 0


            test_acc.append(
                np.sum(correct)/total if total<1000 else np.sum(correct)/1000
            )
            
            if total%1000 == 0:
                print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')
        
            

Iter : 1001, loss: 2.0453, accuracy: 0.1520
Iter : 2001, loss: 2.0427, accuracy: 0.1320
Iter : 3001, loss: 2.1150, accuracy: 0.1380
Iter : 4001, loss: 1.9510, accuracy: 0.1620
Iter : 5001, loss: 2.1833, accuracy: 0.1390
Iter : 6001, loss: 2.2220, accuracy: 0.1270
Iter : 7001, loss: 2.1193, accuracy: 0.1290
Iter : 8001, loss: 2.1950, accuracy: 0.1800
Iter : 9001, loss: 2.0347, accuracy: 0.1390
Iter : 10001, loss: 2.0654, accuracy: 0.1520
Iter : 11001, loss: 2.0432, accuracy: 0.1580
Iter : 12001, loss: 2.0778, accuracy: 0.1530
Iter : 13001, loss: 2.0012, accuracy: 0.1460
Iter : 14001, loss: 2.0276, accuracy: 0.1250
Iter : 15001, loss: 2.1806, accuracy: 0.1310
Iter : 16001, loss: 1.9692, accuracy: 0.1530
Iter : 17001, loss: 1.9317, accuracy: 0.1590
Iter : 18001, loss: 2.2086, accuracy: 0.1660
Iter : 19001, loss: 2.0805, accuracy: 0.1500
Iter : 20001, loss: 2.0411, accuracy: 0.1440
Iter : 21001, loss: 2.2506, accuracy: 0.1310
Iter : 22001, loss: 2.0400, accuracy: 0.1340
Iter : 23001, loss: